Data sources: [US Mortality DataBase](https://usa.mortality.org) / [U.S State Life Tables CSV + TXT 1941-2022](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/19WYUX)<br>
[List of U.S. states and territories by life expectancy](https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_life_expectancy) <i>([alternative design](https://en.wikipedia.org/wiki/User:Lady3mlnm/List_of_U.S._states_and_territories_by_life_expectancy_(alternative)))</i> / [Продолжительность жизни в штатах США](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_штатах_США)<br>
[MapChart](https://www.mapchart.net/usa.html)<br>
<br>
For small exploration:<br>
[results of the 2024 US presidential election](https://en.wikipedia.org/wiki/2024_United_States_presidential_election#Results)

In [2]:
import pandas as pd
from collections import namedtuple
from itertools import product
import math
import re

import sys
sys.path.append("..")
import mal_moduls_private.mal_total as mal

In [3]:
pd.options.display.min_rows=4

In [4]:
YEARS=[2014, 2019, 2020, 2021, 2022]
CREATE_LEGEND_CODE = False

In [5]:
StateInfo = namedtuple('StateInfo', ['en_name', 'en_page', 'ru_name', 'ru_page'])

In [6]:
dd_states = {
    'AL': StateInfo('Alabama', 'Alabama', 'Алаба́ма', 'Алабама'),
    'AK': StateInfo('Alaska', 'Alaska', 'Аля́ска', 'Аляска'),
    'AZ': StateInfo('Arizona', 'Arizona', 'Аризо́на', 'Аризона'),
    'AR': StateInfo('Arkansas', 'Arkansas', 'Арканза́с (Арка́нзас)', 'Арканзас'),
    'CA': StateInfo('California', 'California', 'Калифо́рния', 'Калифорния'),
    'CO': StateInfo('Colorado', 'Colorado', 'Колора́до', 'Колорадо'),
    'CT': StateInfo('Connecticut', 'Connecticut', 'Конне́ктикут', 'Коннектикут'),
    'DE': StateInfo('Delaware', 'Delaware', 'Де́лавэр', 'Делавэр'),
    'DC': StateInfo('Washington, D.C.', 'Washington, D.C.', 'Вашингто́н (Ва́шингтон)', 'Вашингтон'),
    'FL': StateInfo('Florida', 'Florida', 'Флори́да', 'Флорида'),
    'GA': StateInfo('Georgia', 'Georgia (U.S. state)', 'Джо́рджия', 'Джорджия'),
    'HI': StateInfo('Hawaii', 'Hawaii', 'Гава́йи', 'Гавайи'),
    'ID': StateInfo('Idaho', 'Idaho', 'Айда́хо (А́йдахо)', 'Айдахо'),
    'IL': StateInfo('Illinois', 'Illinois', 'Иллино́йс', 'Иллинойс'),
    'IN': StateInfo('Indiana', 'Indiana', 'Индиа́на', 'Индиана'),
    'IA': StateInfo('Iowa', 'Iowa', 'А́йова (Айо́ва)', 'Айова'),
    'KS': StateInfo('Kansas', 'Kansas', 'Ка́нзас (Канза́с)', 'Канзас'),
    'KY': StateInfo('Kentucky', 'Kentucky', 'Кенту́кки', 'Кентукки'),
    'LA': StateInfo('Louisiana', 'Louisiana', 'Луизиа́на', 'Луизиана'),
    'ME': StateInfo('Maine', 'Maine', 'Мэн', 'Мэн (штат)'),
    'MD': StateInfo('Maryland', 'Maryland', 'Мэ́риленд', 'Мэриленд'),
    'MA': StateInfo('Massachusetts', 'Massachusetts', 'Массачу́сетс', 'Массачусетс'),
    'MI': StateInfo('Michigan', 'Michigan', 'Мичига́н', 'Мичиган'),
    'MN': StateInfo('Minnesota', 'Minnesota', 'Миннесо́та', 'Миннесота'),
    'MS': StateInfo('Mississippi', 'Mississippi', 'Миссиси́пи', 'Миссисипи (штат)'),
    'MO': StateInfo('Missouri', 'Missouri', 'Миссу́ри', 'Миссури (штат)'),
    'MT': StateInfo('Montana', 'Montana', 'Монта́на', 'Монтана'),
    'NE': StateInfo('Nebraska', 'Nebraska', 'Небра́ска', 'Небраска'),
    'NV': StateInfo('Nevada', 'Nevada', 'Нева́да', 'Невада'),
    'NH': StateInfo('New Hampshire', 'New Hampshire', 'Нью-Гэ́мпшир', 'Нью-Гэмпшир'),
    'NJ': StateInfo('New Jersey', 'New Jersey', 'Нью-Дже́рси', 'Нью-Джерси'),
    'NM': StateInfo('New Mexico', 'New Mexico', 'Нью-Ме́ксико', 'Нью-Мексико'),
    'NY': StateInfo('New York', 'New York (state)', 'Нью-Йо́рк (штат)', 'Нью-Йорк (штат)'),
    'NC': StateInfo('North Carolina', 'North Carolina', 'Северная Кароли́на', 'Северная Каролина'),
    'ND': StateInfo('North Dakota', 'North Dakota', 'Северная Дако́та', 'Северная Дакота'),
    'OH': StateInfo('Ohio', 'Ohio', 'Ога́йо', 'Огайо'),
    'OK': StateInfo('Oklahoma', 'Oklahoma', 'Оклахо́ма', 'Оклахома'),
    'OR': StateInfo('Oregon', 'Oregon', 'Орего́н', 'Орегон'),
    'PA': StateInfo('Pennsylvania', 'Pennsylvania', 'Пенсильва́ния', 'Пенсильвания'),
    'RI': StateInfo('Rhode Island', 'Rhode Island', 'Род-А́йленд', 'Род-Айленд'),
    'SC': StateInfo('South Carolina', 'South Carolina', 'Южная Кароли́на', 'Южная Каролина'),
    'SD': StateInfo('South Dakota', 'South Dakota', 'Южная Дако́та', 'Южная Дакота'),
    'TN': StateInfo('Tennessee', 'Tennessee', 'Теннесси́', 'Теннесси'),
    'TX': StateInfo('Texas', 'Texas', 'Теха́с', 'Техас'),
    'UT': StateInfo('Utah', 'Utah', 'Ю́та', 'Юта'),
    'VT': StateInfo('Vermont', 'Vermont', 'Вермо́нт', 'Вермонт'),
    'VA': StateInfo('Virginia', 'Virginia', 'Вирги́ния (Вирджи́ния)', 'Виргиния'),
    'WA': StateInfo('Washington (state)', 'Washington (state)', 'Вашингто́н (штат)', 'Вашингтон (штат)'),
    'WV': StateInfo('West Virginia', 'West Virginia', 'Западная Вирги́ния', 'Западная Виргиния'),
    'WI': StateInfo('Wisconsin', 'Wisconsin', 'Виско́нсин', 'Висконсин'),
    'WY': StateInfo('Wyoming', 'Wyoming', 'Вайо́минг', 'Вайоминг')
}

In [7]:
def load_data_single_unit(abbr, name, years, level='States'):
    '''
    The function loads data from disk and constructs dataframe for a separate state or the whole country
    depending on the value of variable 'level'.
    The dataframe contains a single record with life expectancy for given years
    for total population, male, female, and also sex gap between female and male.
    '''
    df_t = pd.read_csv(f"data/{level}/{abbr}/{abbr}_bltper_1x1.csv", sep=',', usecols=['Year', 'Age', 'ex'])
    df_t = df_t.loc[(df_t['Year'].isin(years)) & (df_t['Age'] == '0')] \
               .drop(columns='Age') \
               .set_index('Year') \
               .transpose() \

    df_m = pd.read_csv(f"data/{level}/{abbr}/{abbr}_mltper_1x1.csv", sep=',', usecols=['Year', 'Age', 'ex'])
    df_m = df_m.loc[(df_m['Year'].isin(years)) & (df_m['Age'] == '0')] \
               .drop(columns='Age') \
               .set_index('Year') \
               .transpose() \

    df_f = pd.read_csv(f"data/{level}/{abbr}/{abbr}_fltper_1x1.csv", sep=',', usecols=['Year', 'Age', 'ex'])
    df_f = df_f.loc[(df_f['Year'].isin(years)) & (df_f['Age'] == '0')] \
               .drop(columns='Age') \
               .set_index('Year') \
               .transpose() \

    df = pd.concat([tp[1][tp[0]] for tp in product(years, [df_t, df_m, df_f])],
                   axis='columns')

    df.columns = [f"{tp[0]}_{tp[1]}" for tp in product(years, ['t', 'm', 'f'])]

    for i, year in enumerate(years):
        df.insert(loc=i*4+3, column=f'{year}_fΔm', value=(df[f'{year}_f']-df[f'{year}_m']).round(2))

    df.index = [name]

    return df

# test with a single state
# df = load_data_state('CA', 'California', YEARS)
# df

<br>

In [9]:
# load data for all states
df = pd.concat(load_data_single_unit(abbr, dd.en_name, YEARS) for abbr, dd in dd_states.items())

In [10]:
# combine data for states with data for the whole country
df_country = load_data_single_unit('USA', '– US –', YEARS, level='Nationals')

df = pd.concat([df_country, df.sort_values(by=['2019_t', '2019_m', '2019_f'], ascending=False)])
df.head()

,2014_t,2014_m,2014_f,2014_fΔm,2019_t,2019_m,2019_f,2019_fΔm,2020_t,2020_m,2020_f,2020_fΔm,2021_t,2021_m,2021_f,2021_fΔm,2022_t,2022_m,2022_f,2022_fΔm
– US –,78.89,76.46,81.25,4.79,78.91,76.40,81.43,5.03,77.05,74.32,79.88,5.56,76.45,73.63,79.41,5.78,77.54,74.88,80.28,5.40
Hawaii,81.46,78.63,84.19,5.56,81.60,78.63,84.60,5.97,81.64,78.60,84.71,6.11,80.89,77.85,84.10,6.25,80.88,77.86,84.06,6.20
California,81.04,78.73,83.26,4.53,81.15,78.69,83.60,4.91,79.24,76.44,82.16,5.72,78.58,75.59,81.74,6.15,79.72,76.99,82.55,5.56
New York,80.69,78.27,82.90,4.63,81.06,78.54,83.43,4.89,78.25,75.32,81.19,5.87,79.43,76.68,82.12,5.44,80.08,77.36,82.73,5.37
Minnesota,80.90,78.84,82.89,4.05,80.67,78.45,82.90,4.45,79.29,77.03,81.62,4.59,78.99,76.45,81.65,5.20,79.42,77.13,81.80,4.67


<br>

In [12]:
# data for 2022 I consider not interesting
df.drop(columns=['2022_t', '2022_m', '2022_f', '2022_fΔm'], inplace=True)

In [13]:
# Calculate changes
df.insert(loc=4, column='2014→2019', value=(df['2019_t']-df['2014_t']).round(2))
df.insert(loc=9, column='2019→2020', value=(df['2020_t']-df['2019_t']).round(2))
df.insert(loc=14, column='2020→2021', value=(df['2021_t']-df['2020_t']).round(2))
df.insert(loc=19, column='2019→2021', value=(df['2021_t']-df['2019_t']).round(2))
df.head()

,2014_t,2014_m,2014_f,2014_fΔm,2014→2019,2019_t,2019_m,2019_f,2019_fΔm,2019→2020,2020_t,2020_m,2020_f,2020_fΔm,2020→2021,2021_t,2021_m,2021_f,2021_fΔm,2019→2021
– US –,78.89,76.46,81.25,4.79,0.02,78.91,76.40,81.43,5.03,-1.86,77.05,74.32,79.88,5.56,-0.60,76.45,73.63,79.41,5.78,-2.46
Hawaii,81.46,78.63,84.19,5.56,0.14,81.60,78.63,84.60,5.97,0.04,81.64,78.60,84.71,6.11,-0.75,80.89,77.85,84.10,6.25,-0.71
California,81.04,78.73,83.26,4.53,0.11,81.15,78.69,83.60,4.91,-1.91,79.24,76.44,82.16,5.72,-0.66,78.58,75.59,81.74,6.15,-2.57
New York,80.69,78.27,82.90,4.63,0.37,81.06,78.54,83.43,4.89,-2.81,78.25,75.32,81.19,5.87,1.18,79.43,76.68,82.12,5.44,-1.63
Minnesota,80.90,78.84,82.89,4.05,-0.23,80.67,78.45,82.90,4.45,-1.38,79.29,77.03,81.62,4.59,-0.30,78.99,76.45,81.65,5.20,-1.68


<br>

In [15]:
# just for interest, explore results: determine regions with max and min values, and also look at specific regions
mal.min_and_max_values(df[['2014_t', '2014→2019', '2019_t', '2019→2020', '2020_t', '2020→2021', '2021_t', '2019→2021']],
                       row_center=['– US –'], nmb=5, max_lng=13)

Number of records: 52


,2014_t,2014→2019,2019_t,2019→2020,2020_t,2020→2021,2021_t,2019→2021
max,81.46 -Hawaii,0.37 -New York,81.6 -Hawaii,0.04 -Hawaii,81.64 -Hawaii,1.32 -New Jersey,80.89 -Hawaii,-0.71 -Hawaii
max_2,81.04 -California,0.36 -Texas,81.15 -California,-0.4 -New Hampshire,79.37 -Washington (…,1.18 -New York,79.82 -Massachusetts,-0.81 -Massachusetts
max_3,80.97 -Connecticut,0.34 -Idaho,81.06 -New York,-0.47 -Maine,79.29 -Minnesota,0.7 -Connecticut,79.43 -New York,-0.96 -New Hampshire
max_4,80.9 -Minnesota,0.28 -Louisiana,80.67 -Minnesota,-0.73 -Vermont,79.29 -Vermont,0.6 -North Dakota,79.38 -Connecticut,-0.99 -Rhode Island
max_5,80.69 -New York,0.27 -Utah,80.63 -Massachusetts,-0.81 -Oregon,79.27 -New Hampshire,0.56 -Massachusetts,79.25 -New Jersey,-1.15 -New Jersey
– US –,– 78.89 –,– 0.02 –,– 78.91 –,– -1.86 –,– 77.05 –,– -0.6 –,– 76.45 –,– -2.46 –
min_5,75.62 -Oklahoma,-0.52 -Ohio,75.77 -Tennessee,-2.61 -Mississippi,73.69 -Kentucky,-1.42 -Oklahoma,72.44 -Tennessee,-3.56 -Louisiana
min_4,75.61 -Louisiana,-0.53 -West Virginia,75.62 -Kentucky,-2.7 -Louisiana,73.52 -Alabama,-1.46 -Tennessee,72.33 -Louisiana,-3.57 -Mississippi
min_3,75.44 -Alabama,-0.54 -North Dakota,75.48 -Alabama,-2.73 -Arizona,73.19 -Louisiana,-1.47 -New Mexico,72.18 -Alabama,-3.57 -West Virginia
min_2,75.21 -West Virginia,-0.59 -Maine,74.68 -West Virginia,-2.81 -New York,72.99 -West Virginia,-1.88 -West Virginia,71.11 -West Virginia,-3.86 -Arizona


In [16]:
mal.min_and_max_values(df[['2014_m', '2014_t', '2014_f', '2014→2019', '2019_m', '2019_t', '2019_f']],
                       row_center=['– US –'], nmb=5, max_lng=13)

Number of records: 52


,2014_m,2014_t,2014_f,2014→2019,2019_m,2019_t,2019_f
max,78.84 -Minnesota,81.46 -Hawaii,84.19 -Hawaii,0.37 -New York,78.69 -California,81.6 -Hawaii,84.6 -Hawaii
max_2,78.73 -California,81.04 -California,83.26 -California,0.36 -Texas,78.63 -Hawaii,81.15 -California,83.6 -California
max_3,78.63 -Hawaii,80.97 -Connecticut,83.17 -Connecticut,0.34 -Idaho,78.54 -New York,81.06 -New York,83.43 -New York
max_4,78.61 -Connecticut,80.9 -Minnesota,82.95 -Massachusetts,0.28 -Louisiana,78.45 -Minnesota,80.67 -Minnesota,83.08 -Connecticut
max_5,78.27 -New York,80.69 -New York,82.9 -New York,0.27 -Utah,78.19 -Utah,80.63 -Massachusetts,83.05 -Massachusetts
– US –,– 76.46 –,– 78.89 –,– 81.25 –,– 0.02 –,– 76.4 –,– 78.91 –,– 81.43 –
min_5,73.11 -Kentucky,75.62 -Oklahoma,78.26 -Louisiana,-0.52 -Ohio,73.05 -Louisiana,75.77 -Tennessee,78.47 -Oklahoma
min_4,72.93 -Louisiana,75.61 -Louisiana,78.08 -Alabama,-0.53 -West Virginia,72.94 -Tennessee,75.62 -Kentucky,78.46 -Alabama
min_3,72.74 -Alabama,75.44 -Alabama,78.0 -Oklahoma,-0.54 -North Dakota,72.51 -Alabama,75.48 -Alabama,78.19 -Kentucky
min_2,72.55 -West Virginia,75.21 -West Virginia,77.96 -West Virginia,-0.59 -Maine,72.11 -West Virginia,74.68 -West Virginia,77.75 -Mississippi


<br>
<br>
<br>

In [18]:
# create 'dd_replacement' dictionary used in previous script

dd_replacement = {}

dd_replacement['– US –'] = {'en': ('United States', ''), 'ru': ('Соединённые Штаты', '')}

for state in dd_states.values():
    dd_replacement[state.en_name] = {'en': (state.en_name, state.en_page), 'ru': (state.ru_name, state.ru_page)}

In [19]:
# create code for placing info in Wikipedia
def create_table_v1(df, file_header, lang='ru'):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    # def chval(x, prec=1, *, add_par=''):  # change_value
    #     return f'style="padding-right:4ex;{add_par}"|—' if math.isnan(x) else \
    #            f'style="padding-right:4ex;color:darkgreen;{add_par}"|{x:0.{prec}f}' if x>0 else \
    #            f'style="padding-right:4ex;color:crimson;{add_par}"|−{-x:0.{prec}f}' if x<0 else \
    #            f'style="padding-right:4ex;color:darkgray;{add_par}"|{x:0.{prec}f}'
    
    # def chval_bold(x, prec=1, *, add_par=''):  # change_value
    #     return '—' if math.isnan(x) else \
    #            f'style="padding-right:4ex;color:darkgreen;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
    #            f'style="padding-right:4ex;color:crimson;{add_par}"|\'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
    #            f'style="padding-right:4ex;color:darkgray;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\''
    
    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]    
        if ser.name == '– US –':
            st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2019_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2019_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2019_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:1.5ex;border-left-width:2px;"| \'\'\'{if_value(ser["2019→2020"])}\'\'\' ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2020_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2020_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2020_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2020_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:1.5ex;border-left-width:2px;"| \'\'\'{if_value(ser["2020→2021"])}\'\'\' ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2021_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2021_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2021_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:1.5ex;border-left-width:2px;"| \'\'\'{if_value(ser["2019→2021"])}\'\'\''
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2019_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2019_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2019_fΔm"])} ' + \
                  f'||style="padding-right:1.5ex;border-left-width:2px;"| {if_value(ser["2019→2020"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2020_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2020_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2020_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2020_fΔm"])} ' + \
                  f'||style="padding-right:1.5ex;border-left-width:2px;"| {if_value(ser["2020→2021"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2021_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2021_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2021_fΔm"])} ' + \
                  f'||style="padding-right:1.5ex;border-left-width:2px;"| {if_value(ser["2019→2021"])}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —') \
           .replace(';"| \'\'\'—\'\'\'', ';color:silver;"| \'\'\'—\'\'\'')

    return st


table_code = create_table_v1(df, file_header='LE_header -2021, v1, ru.txt', lang='ru')
with open('output/Table code for LE -v1 -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

table_code = create_table_v1(df, file_header='LE_header -2021, v1, en.txt', lang='en')
with open('output/Table code for LE -v1 -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [20]:
# create code for placing info in Wikipedia
def create_table_v2(df, file_header, lang='ru'):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    def chval(x, prec=2, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;border-left-width:2px;{add_par}"| —' if math.isnan(x) else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgreen;{add_par}"| {x:0.{prec}f}' if x>0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:crimson;{add_par}"| −{-x:0.{prec}f}' if x<0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgray;{add_par}"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;border-left-width:2px;color:darkgreen;{add_par}"| \'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgreen;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:crimson;{add_par}"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgray;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\''
    
    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]    
        if ser.name == '– US –':
            st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2014_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2014_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2014_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2014_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2019_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2019_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2019_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2021"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2021_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2021_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2021_fΔm"])}\'\'\''
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2014_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2014_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2014_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2014_fΔm"])} ' + \
                  f'||{chval(ser["2014→2019"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2019_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2019_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2019_fΔm"])} ' + \
                  f'||{chval(ser["2019→2021"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2021_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2021_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2021_fΔm"])}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —') \
           .replace(';"| \'\'\'—\'\'\'', ';color:silver;"| \'\'\'—\'\'\'')

    return st


table_code = create_table_v2(df, file_header='LE_header -2021, v2, ru.txt', lang='ru')
with open('output/Table code for LE -v2 -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

table_code = create_table_v2(df, file_header='LE_header -2021, v2, en.txt', lang='en')
with open('output/Table code for LE -v2 -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [21]:
# create code for placing info in Wikipedia
def create_table_v3(df, file_header, lang='ru'):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    def chval(x, prec=2, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;border-left-width:2px;{add_par}"| —' if math.isnan(x) else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgreen;{add_par}"| {x:0.{prec}f}' if x>0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:crimson;{add_par}"| −{-x:0.{prec}f}' if x<0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgray;{add_par}"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;border-left-width:2px;color:darkgreen;{add_par}"| \'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgreen;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:crimson;{add_par}"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="padding-right:1.5ex;border-left-width:2px;color:darkgray;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\''
    
    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]    
        if ser.name == '– US –':
            st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2014_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2014_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2014_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2014_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2019_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2019_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2019_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2020"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2020_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2020_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2020_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2020_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2020→2021"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2021_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2021_f"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["2021_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2021"])}'
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2014_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2014_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2014_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2014_fΔm"])} ' + \
                  f'||{chval(ser["2014→2019"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2019_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2019_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2019_fΔm"])} ' + \
                  f'||{chval(ser["2019→2020"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2020_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2020_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2020_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2020_fΔm"])} ' + \
                  f'||{chval(ser["2020→2021"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2021_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2021_f"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["2021_fΔm"])} ' + \
                  f'||{chval(ser["2019→2021"])}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —') \
           .replace(';"| \'\'\'—\'\'\'', ';color:silver;"| \'\'\'—\'\'\'')

    return st


# table_code = create_table_v3(df, file_header='LE_header -2021, v3, ru.txt', lang='ru')
# with open('output/Table code for LE -v3 -ru.txt', 'w', encoding="utf-8") as fh:
#     fh.write(table_code)

table_code = create_table_v3(df, file_header='LE_header -2021, v3, en.txt', lang='en')
with open('output/Table code for LE -v3 -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

<br>
<br>
<br>
<hr>

<h3>Maps creation</h3>

In [23]:
SELECTED_YEAR = 2019

In [24]:
CountryGroup = namedtuple('CountryGroup', ['group_label', 'color', 'countries'])

In [25]:
# for state in sorted(df.index.to_list()):
#     print(f"               '{state}' : ''")

In [26]:
df_map = df.copy()                                    \
           .drop(['– US –']) \
           .rename(index={
               'Alabama' : 'AL',
               'Alaska' : 'AK',
               'Arizona' : 'AZ',
               'Arkansas' : 'AR',
               'California' : 'CA',
               'Colorado' : 'CO',
               'Connecticut' : 'CT',
               'Delaware' : 'DE',
               'Washington, D.C.' : 'DC',
               'Florida' : 'FL',
               'Georgia' : 'GA',
               'Hawaii' : 'HI',
               'Idaho' : 'ID',
               'Illinois' : 'IL',
               'Indiana' : 'IN',
               'Iowa' : 'IA',
               'Kansas' : 'KS',
               'Kentucky' : 'KY',
               'Louisiana' : 'LA',
               'Maine' : 'ME',
               'Maryland' : 'MD',
               'Massachusetts' : 'MA',
               'Michigan' : 'MI',
               'Minnesota' : 'MN',
               'Mississippi' : 'MS',
               'Missouri' : 'MO',
               'Montana' : 'MT',
               'Nebraska' : 'NE',
               'Nevada' : 'NV',
               'New Hampshire' : 'NH',
               'New Jersey' : 'NJ',
               'New Mexico' : 'NM',
               'New York' : 'NY',
               'North Carolina' : 'NC',
               'North Dakota' : 'ND',
               'Ohio' : 'OH',
               'Oklahoma' : 'OK',
               'Oregon' : 'OR',
               'Pennsylvania' : 'PA',
               'Rhode Island' : 'RI',
               'South Carolina' : 'SC',
               'South Dakota' : 'SD',
               'Tennessee' : 'TN',
               'Texas' : 'TX',
               'Utah' : 'UT',
               'Vermont' : 'VT',
               'Virginia' : 'VA',
               'Washington (state)' : 'WA',
               'West Virginia' : 'WV',
               'Wisconsin' : 'WI',
               'Wyoming' : 'WY'
           })

df_map.head(3).fillna('')

,2014_t,2014_m,2014_f,2014_fΔm,2014→2019,2019_t,2019_m,2019_f,2019_fΔm,2019→2020,2020_t,2020_m,2020_f,2020_fΔm,2020→2021,2021_t,2021_m,2021_f,2021_fΔm,2019→2021
HI,81.46,78.63,84.19,5.56,0.14,81.60,78.63,84.60,5.97,0.04,81.64,78.60,84.71,6.11,-0.75,80.89,77.85,84.10,6.25,-0.71
CA,81.04,78.73,83.26,4.53,0.11,81.15,78.69,83.60,4.91,-1.91,79.24,76.44,82.16,5.72,-0.66,78.58,75.59,81.74,6.15,-2.57
NY,80.69,78.27,82.90,4.63,0.37,81.06,78.54,83.43,4.89,-2.81,78.25,75.32,81.19,5.87,1.18,79.43,76.68,82.12,5.44,-1.63


In [27]:
dd_legend_main = {
    '82.00–82.24' : '007800',
    '81.75–81.99' : '008800',
    '81.50–81.74' : '009800',
    '81.25–81.49' : '00a700',
    '81.00–81.24' : '00b800',
    '80.75–80.99' : '00cb00',
    '80.50–80.74' : '00e000',
    '80.25–80.49' : '00f000',
    '80.00–80.24' : '00ff00',
    '79.75–79.99' : '88ff00',
    '79.50–79.74' : 'b8ff00',
    '79.25–79.49' : 'deff00',
    '79.00–79.24' : 'ffff00',
    '78.75–78.99' : 'ffef00',
    '78.50–78.74' : 'ffe000',
    '78.25–78.49' : 'ffce00',
    '78.00–78.24' : 'ffbc00',
    '77.75–77.99' : 'ffac00',
    '77.50–77.74' : 'ff9c00',
    '77.25–77.49' : 'ff8700',
    '77.00–77.24' : 'ff7400',
    '76.75–76.99' : 'ff5200',
    '76.50–76.74' : 'ff0000',
    '76.25–76.49' : 'e10000',
    '76.00–76.24' : 'c70000',
    '75.75–75.99' : 'af0000',
    '75.50–75.74' : '9e0000',
    '75.25–75.49' : '8b0000',
    '75.00–75.24' : '790000',
    '74.75–74.99' : '670000',
    '74.50–74.74' : '580000',
}

def fn_create_legend_code(dd_legend):
    for k, v in dd_legend.items():
        print(f"{{{{Legend|#{v}|{k}}}}}")

if CREATE_LEGEND_CODE:
    fn_create_legend_code(dd_legend_main)

In [28]:
df_grouped = mal.bin_values_in_dataframe(df_map, f"{SELECTED_YEAR}_t", prec=2, step = 0.25)

# df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '70.9–71.4' if st < '71.5–71.9' else st)
# df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '80.5–81.0' if st > '80.0–80.4' else st)

df_grouped.head()

Range: 74.54 – 81.60   (MS – HI)
Number of groups: 20
Number of values: 51


,2019_t,group_label
HI,81.60,81.50–81.74
CA,81.15,81.00–81.24
NY,81.06,81.00–81.24
MN,80.67,80.50–80.74
MA,80.63,80.50–80.74


In [29]:
def extract_indexes(subdf, dd_legend):
    group_label = subdf['group_label'].iloc[0]
    countries = subdf.index.to_list()
    color = (dd_legend[group_label])
    
    ls_grouping.append(CountryGroup(group_label=group_label, countries=countries, color=color))

    return pd.Series([color, countries], index=['color', 'regions'])


ls_grouping = []
df_grouped = df_grouped.groupby(['group_label'])[['group_label']].apply(extract_indexes, dd_legend = dd_legend_main).loc[::-1]

df_grouped.head()

,color,regions
group_label,,
81.50–81.74,009800,[HI]
81.00–81.24,00b800,"[CA, NY]"
80.50–80.74,00e000,"[MN, MA, CT]"
80.25–80.49,00f000,"[NJ, CO]"
80.00–80.24,00ff00,"[WA, VT]"


In [30]:
def create_map_code(ls_grouping, title='', legend_color='#000'):  # (ls_grouping, title='', legend_color='#000'):
    false, true = False, True

    jo = {
        "groups": { },
        "title": title,
        "hidden": ["Guam", "Northern_Mariana_Islands", "Puerto_Rico", "American_Samoa", "United_States_Virgin_Islands", "Marshall_Islands", "Palau", "Federated_States_of_Micronesia", "MN_S", "MN_N", "OH_E", "OH_W", "LA_N", "LA_S", "ID_S", "ID_N", "NM_N", "NM_S", "CO_S", "CO_N", "WY_N", "WY_S", "MT_S", "MT_N", "VA_S", "VA_N", "KS_W", "KS_E", "ND_W", "ND_E", "SD_W", "SD_E", "NE_W", "NE_E", "CO_W", "CO_E", "WY_W", "WY_E", "MT_C", "MT_W", "MT_E", "SC_W", "SC_E", "ME_N", "ME_S", "MI_N", "MD_E", "NJ_C", "NJ_N", "PA_E", "PA_S", "TX_W", "TX_S", "PA_N", "PA_W", "FL_S", "FL_C", "FL_N", "MI_E", "MI_W", "VA_E", "WV_S", "WV_N", "OH_S", "OH_N", "IN_S", "IN_N", "IL_C", "IL_N", "WI_W", "NC_W", "NC_E", "MO_W", "MO_E", "GA_S", "GA_N", "KY_W", "KY_E", "AL_S", "AL_N", "LA_E", "LA_W", "MS_S", "MS_N", "IA_S", "IA_N", "OK_E", "OK_W", "UT_S", "UT_N", "AZ_S", "AZ_N", "NV_S", "NV_N", "OR_E", "OR_W", "WA_E", "WA_W", "CA_S", "CA_N", "TN_C", "TN_E", "AR_S", "AR_N", "NY_N", "TX_E", "TX_N", "WI_E", "IL_S", "TN_W", "VA_W", "NY_S", "DE_N", "DE_S", "NJ_S", "MD_W", "MI_S", "IN_E", "IN_W"],
        "background": "#ffffff",
        "borders": "#000",
        "legendFont": "Century Gothic",
        "legendFontColor": "#000",
        "legendBorderColor": "#00000000",
        "legendBgColor": "#00000000",
        "legendWidth": 100,
        "legendBoxShape": "square",
        "areBordersShown": true,
        "defaultColor": "#d1dbdd",
        "labelsColor": "#000000",
        "labelsFont": "Arial",
        "strokeWidth": "medium",
        "areLabelsShown": true,
        "uncoloredScriptColor": "#ffff33",
        "zoomLevel": "1.00",
        "zoomX": "0.00",
        "zoomY": "0.00",
        "v6": true,
        "page": "usa",
        "usTerritoriesShown": false,
        "usFasShown": false,
        "splitStates": {},
        "legendPosition": "custom",
        "legendX": 860,
        "legendY": 180,
        "legendSize": "small",
        "legendTranslateX": "0.00",
        "legendStatus": "show",
        "scalingPatterns": true,
        "legendRowsSameColor": true,
        "legendColumnCount": 1
    }

    for group_label, color, regions in ls_grouping[::-1]:
        jo["groups"][f"#{color}"] = {"label": group_label.replace('.00', '.0').replace('.25', '¼').replace('.50', '.5').replace('.75', '¾').replace('–', ' - '),
                                     "paths": [region.replace(' ', '_') for region in regions]}       

    return jo


jo = create_map_code(ls_grouping, title=f'{SELECTED_YEAR}', legend_color='#333333')

pretty_jo = json.dumps(jo, indent=2, ensure_ascii=False)

with open(f"output/map_USMDB_JSON -{SELECTED_YEAR}.txt", 'w', encoding="utf-8") as fh:
    fh.write(pretty_jo)

<br />
<br />

<h4>Statistics for males</h4>

In [32]:
dd_legend_male = {
    '78.75–78.99' : '020f19',
    '78.50–78.74' : '042136',
    '78.25–78.49' : '062f4c',
    '78.00–78.24' : '073a5e',
    '77.75–77.99' : '08436e',
    '77.50–77.74' : '0a4e7f',
    '77.25–77.49' : '0b578d',
    '77.00–77.24' : '0c609c',
    '76.75–76.99' : '0d68a9',
    '76.50–76.74' : '0e71b7',
    '76.25–76.49' : '107ac6',
    '76.00–76.24' : '1182d4',
    '75.75–75.99' : '128be1',
    '75.50–75.74' : '1c95ec',
    '75.25–75.49' : '2e9eee',
    '75.00–75.24' : '40a6ef',
    '74.75–74.99' : '51aef0',
    '74.50–74.74' : '61b5f2',
    '74.25–74.49' : '6ebcf3',
    '74.00–74.24' : '80c4f4',
    '73.75–73.99' : '92ccf6',
    '73.50–73.74' : 'a2d4f7',
    '73.25–73.49' : 'b3dbf8',
    '73.00–73.24' : 'c2e3fa',
    '72.75–72.99' : 'd4ebfb',
    '72.50–72.74' : 'e4f2fc',
    '72.25–72.49' : 'edf7fd',
    '72.00–72.24' : 'f4fafe',
    # '71.75–71.99' : 'f8fbfe',
    '<72' : 'f8fbfe'
}

if CREATE_LEGEND_CODE:
    fn_create_legend_code(dd_legend_male)

In [33]:
df_grouped = mal.bin_values_in_dataframe(df_map, f"{SELECTED_YEAR}_m", prec=2, step = 0.25)

df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '<72' if st < '72.00–72.24' else st)
# df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '80.5–81.0' if st > '80.0–80.4' else st)

df_grouped.head()

Range: 71.37 – 78.69   (MS – CA)
Number of groups: 24
Number of values: 51


,2019_m,group_label
CA,78.69,78.50–78.74
HI,78.63,78.50–78.74
NY,78.54,78.50–78.74
MN,78.45,78.25–78.49
UT,78.19,78.00–78.24


In [34]:
ls_grouping = []
df_grouped = df_grouped.groupby(['group_label'])[['group_label']].apply(extract_indexes, dd_legend = dd_legend_male).loc[::-1]

df_grouped.head()

,color,regions
group_label,,
<72,f8fbfe,[MS]
78.50–78.74,042136,"[CA, HI, NY]"
78.25–78.49,062f4c,[MN]
78.00–78.24,073a5e,"[UT, MA, WA, CO]"
77.75–77.99,08436e,"[CT, NJ, ID]"


In [35]:
jo = create_map_code(ls_grouping, title=f'{SELECTED_YEAR}: ♂', legend_color='#333333')  # legend_color='#0000bf'

pretty_jo = json.dumps(jo, indent=2, ensure_ascii=False)

with open(f"output/map_USMDB_JSON -{SELECTED_YEAR}, male.txt", 'w', encoding="utf-8") as fh:
    fh.write(pretty_jo)

<br />
<br />

<h4>Statistics for females</h4>

In [37]:
dd_legend_female = {
    '84.75–84.99' : '1e0204',
    '84.50–84.74' : '3f050a',
    '84.25–84.49' : '5a070e',
    '84.00–84.24' : '700911',
    '83.75–83.99' : '860a15',
    '83.50–83.74' : '9c0c18',
    '83.25–83.49' : 'ad0e1b',
    '83.00–83.24' : 'bd0f25',
    '82.75–82.99' : 'cd1030',
    '82.50–82.74' : 'df1234',
    '82.25–82.49' : 'ec2042',
    '82.00–82.24' : 'ef3b59',
    '81.75–81.99' : 'f04f6a',
    '81.50–81.74' : 'f15d76',
    '81.25–81.49' : 'f26a80',
    '81.00–81.24' : 'f3768b',
    '80.75–80.99' : 'f48295',
    '80.50–80.74' : 'f58fa0',
    '80.25–80.49' : 'f69aaa',
    '80.00–80.24' : 'f7a4b2',
    '79.75–79.99' : 'f8acb9',
    '79.50–79.74' : 'f9b5c0',
    '79.25–79.49' : 'f9bec5',
    '79.00–79.24' : 'fac7cd',
    '78.75–78.99' : 'fbcfd3',
    '78.50–78.74' : 'fbd7da',
    '78.25–78.49' : 'fcdfe2',
    '78.00–78.24' : 'fde7e9',
    '77.75–77.99' : 'fdeeef',
    '77.50–77.74' : 'fef4f4',
    '77.25–77.49' : 'fef8f9'
}

if CREATE_LEGEND_CODE:
    fn_create_legend_code(dd_legend_female)

In [38]:
df_grouped = mal.bin_values_in_dataframe(df_map, f"{SELECTED_YEAR}_f", prec=2, step = 0.25)

# df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '<72' if st < '72.00–72.24' else st)
# df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '80.5–81.0' if st > '80.0–80.4' else st)

df_grouped.head()

Range: 77.41 – 84.60   (WV – HI)
Number of groups: 23
Number of values: 51


,2019_f,group_label
HI,84.60,84.50–84.74
CA,83.60,83.50–83.74
NY,83.43,83.25–83.49
CT,83.08,83.00–83.24
MA,83.05,83.00–83.24


In [39]:
ls_grouping = []
df_grouped = df_grouped.groupby(['group_label'])[['group_label']].apply(extract_indexes, dd_legend = dd_legend_female).loc[::-1]

df_grouped.head()

,color,regions
group_label,,
84.50–84.74,3f050a,[HI]
83.50–83.74,9c0c18,[CA]
83.25–83.49,ad0e1b,[NY]
83.00–83.24,bd0f25,"[CT, MA]"
82.75–82.99,cd1030,"[MN, NJ]"


In [40]:
jo = create_map_code(ls_grouping, title=f'{SELECTED_YEAR}: ♀', legend_color='#333333')  # legend_color='#0000bf'

pretty_jo = json.dumps(jo, indent=2, ensure_ascii=False)

with open(f"output/map_USMDB_JSON -{SELECTED_YEAR}, female.txt", 'w', encoding="utf-8") as fh:
    fh.write(pretty_jo)